In [1]:
from selenium import webdriver
from selenium.webdriver.chrome.options import Options
from bs4 import BeautifulSoup
import time
from urls import urls

import re
import json

from pymongo import MongoClient
import os
from dotenv import load_dotenv

from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough, RunnableMap, RunnableLambda
from langchain_anthropic import ChatAnthropic
from langchain.memory import ConversationBufferMemory
from langchain_core.output_parsers import StrOutputParser

from langchain_groq import ChatGroq

from typing import Literal, Dict
from typing import Literal, Dict, Optional
from langchain.output_parsers import PydanticOutputParser, StructuredOutputParser, OutputFixingParser
from pydantic import BaseModel, Field


class FeatureDetails(BaseModel):
    rating: Literal["good", "bad", "neutral"]
    details: str

class InsurancePolicy(BaseModel):
    insurance_name: str
    provider: str
    features: Dict[str, FeatureDetails] = Field(
        description="Dictionary of features and their ratings/details"
    )

parser = PydanticOutputParser(pydantic_object=InsurancePolicy)
format_instructions = parser.get_format_instructions()

load_dotenv()

True

In [ ]:
def scrape_icici_lombard(url):
    # URL to scrape
    # Configure Chrome options
    options = Options()
    options.add_argument('--headless')
    options.add_argument('--disable-gpu')
    options.add_argument('--no-sandbox')
    options.add_argument('--disable-dev-shm-usage')
    options.add_argument('--user-agent=Mozilla/5.0 (Windows NT 10.0; Win64; x64) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/91.0.4472.124 Safari/537.36')
    
    # Initialize the browser
    browser = webdriver.Chrome(options=options)
    
    try:
        # Access the website
        browser.get(url)
        
        # Wait for page to load
        time.sleep(5)
        
        # Get page source
        html_content = browser.page_source
        
        # Parse HTML with BeautifulSoup
        soup = BeautifulSoup(html_content, 'html.parser')
        
        # Extract all text from the page
        all_text = soup.get_text(separator=' ', strip=True)

        print(f"Successfully scraped {len(all_text)} characters of text")
        
        return all_text
        
    except Exception as e:
        print(f"Error: {str(e)}")
        return None
    finally:
        # Close the browser
        browser.quit()
        print("Browser closed")

In [3]:
out=scrape_icici_lombard(r'https://joinditto.in/health-insurance/bajaj-allianz/')

Successfully scraped 13096 characters of text
Browser closed


In [5]:
print(out)

Bajaj Allianz: Health Insurance Review Ditto Buy Insurance Open menu Health Insurance Life Insurance Claims Renew your policy Careers Buy Insurance Overview Health Insurance Plans Claim Process Renewal Reviews Customer Care Bajaj Allianz Overview Health Insurance Plans Claim Process Renewal Reviews Customer Care Health Insurance Bajaj Allianz Bajaj Allianz General Insurance is a joint venture between Bajaj Finserv Limited (of Bajaj Group) and Allianz SE, a German financial services company. Founded in 2001, the company offers various health, travel, motor, and other general insurance products. It has health insurance products tailor made for individuals, senior citizens, families, and international travelers. It's health insurance has a network of cashless hospitals, an in-house health administration team. It provides 60mins claim settlements. It offers health insurance policies with life time renewability. Claim Settlement Ratio - 97 % (average of last 3 years) Network Hospitals - 12,

In [21]:
mixtral = 'mixtral-8x7b-32768'
llama = 'llama3-70b-8192'

model = ChatGroq(temperature=0, model_name=llama) 

In [63]:
def extract_policy_names(text):
    match = re.search(r'\[(.*?)\]', text)
    
    if not match:
        return []

    content = match.group(1)

    items = content.split(',')

    cleaned_items = []
    for item in items:
        clean_item = item.strip().strip("'").strip('"')
        if clean_item:  # Only add non-empty items
            cleaned_items.append(clean_item)
    
    return cleaned_items

In [48]:
template = ChatPromptTemplate.from_messages([
    ("system", """
        Based on the text given by human, extract ONLY health insurance policy names.  I just need policy names  in output
        Each policy name should be in lowercase with hyphens separating words.
     Please give ONLY output without any explanation and verbose.

     Sample output:
     ['policy1', policy2, policy3,.......]
        
    """),
    ("human", "{question}")
])

# Then use robust_parser instead of parser in your chain
chain = (
    RunnableMap({
        "question": lambda x: x["question"],
    })
    | template 
    | model 
)

In [58]:

final = []
import time

for url in urls:
    data = scrape_icici_lombard(url)
    print("model is parsing")
    response = chain.invoke({"question": data})
    print(response.content)
    final.append(response.content)
    print('=================================================================')

Successfully scraped 7635 characters of text
Browser closed
model is parsing
[platinum-health, standard-health, aditya-birla-health-insurance, zuno-health-insurance, hdfc-ergo-health-insurance]
Successfully scraped 9083 characters of text
Browser closed
model is parsing
[easy-health-exclusive, optima-restore, optima-senior, energy-silver, easy-health-premium, energy-gold, easy-health-standard]
Successfully scraped 8430 characters of text
Browser closed
model is parsing
[smart-super-health-assure, smart-super-top-up, smart-super-health]
Successfully scraped 9526 characters of text
Browser closed
model is parsing
['digit-insurance-comfort-option', 'digit-worldwide-treatment-plan', 'digit-double-wallet-plan', 'digit-insurance-smart-option', 'digit-infinity-wallet-plan']
Successfully scraped 12336 characters of text
Browser closed
model is parsing
Here is the list of health insurance policy names in lowercase with hyphens separating words:

['ihealth', 'max-protect-classic', 'health-shield

In [65]:
final_policy_names = []
for policy in final:
    lst = extract_policy_names(policy)
    final_policy_names.append(lst)

In [83]:
final_policy_names

[['platinum-health',
  'standard-health',
  'aditya-birla-health-insurance',
  'zuno-health-insurance',
  'hdfc-ergo-health-insurance'],
 ['easy-health-exclusive',
  'optima-restore',
  'optima-senior',
  'energy-silver',
  'easy-health-premium',
  'energy-gold',
  'easy-health-standard'],
 ['smart-super-health-assure', 'smart-super-top-up', 'smart-super-health'],
 ['digit-insurance-comfort-option',
  'digit-worldwide-treatment-plan',
  'digit-double-wallet-plan',
  'digit-insurance-smart-option',
  'digit-infinity-wallet-plan'],
 ['ihealth',
  'max-protect-classic',
  'health-shield-360-retail',
  'health-shield-360',
  'health-advantedge',
  'max-protect-premium',
  'golden-shield',
  'health-booster-super-top-up',
  'health-elite-plus',
  'ihealth-plus',
  'elevate'],
 ['prohealth-plus',
  'prohealth-preferred',
  'prohealth-premier',
  'prohealth-select',
  'prohealth-prime-senior-classic',
  'prohealth-prime-advantage',
  'prohealth-prime-senior-elite',
  'prohealth-accumulate',
 

In [88]:
global_poly_names = []
for i in range(len(urls)):
    main_url = urls[i]

    policy_list = final_policy_names[i]

    for p in policy_list:
        global_poly_names.append(main_url+p)



In [ ]:
https://joinditto.in/health-insurance/apollo-munich/

In [89]:
global_poly_names

['https://joinditto.in/health-insurance/acko/platinum-health',
 'https://joinditto.in/health-insurance/acko/standard-health',
 'https://joinditto.in/health-insurance/acko/aditya-birla-health-insurance',
 'https://joinditto.in/health-insurance/acko/zuno-health-insurance',
 'https://joinditto.in/health-insurance/acko/hdfc-ergo-health-insurance',
 'https://joinditto.in/health-insurance/apollo-munich/easy-health-exclusive',
 'https://joinditto.in/health-insurance/apollo-munich/optima-restore',
 'https://joinditto.in/health-insurance/apollo-munich/optima-senior',
 'https://joinditto.in/health-insurance/apollo-munich/energy-silver',
 'https://joinditto.in/health-insurance/apollo-munich/easy-health-premium',
 'https://joinditto.in/health-insurance/apollo-munich/energy-gold',
 'https://joinditto.in/health-insurance/apollo-munich/easy-health-standard',
 'https://joinditto.in/health-insurance/bharti-axa/smart-super-health-assure',
 'https://joinditto.in/health-insurance/bharti-axa/smart-super-to

In [56]:
final

[]